# Day 1: Traditional NLP Basics

Moving from Week 3's deep learning mechanics into actual NLP preprocessing techniques -- normalization, tokenization, stopwords, stemming vs lemmatization, n-grams, and finally TF-IDF, which we've never actually built ourselves even though it's the classic alternative to the `CountVectorizer` bag-of-words approach we've used since Week 2.

Same 278-row news dataset as the rest of this project.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import string
from nltk.tokenize import word_tokenize

df = pd.read_csv("news_dataset.csv")

X_train_text, X_test_text, y_train_labels, y_test_labels = train_test_split(
    df["Title"], df["Category"], test_size=0.2, random_state=42
)

print("Train shape:", X_train_text.shape)
print("Test shape:", X_test_text.shape)

Train shape: (222,)
Test shape: (56,)


In [2]:
def normalize_and_tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation)) ##builds a translation table that maps punctuation characters to nothing and .translate applies it 
    return word_tokenize(text) ##the actual NLTK tokenizer that understands contractions and edgecases that are missed by whitespace

sample = X_train_text.iloc[0]
print("Original:", sample)
print("Normalize + tokenized:", normalize_and_tokenize(sample))

Original: China's Xi to bring large CEO delegation on US visit, sources say
Normalize + tokenized: ['chinas', 'xi', 'to', 'bring', 'large', 'ceo', 'delegation', 'on', 'us', 'visit', 'sources', 'say']


Observation on the output: China's became 'chinas' instead of 'china' + "'s" -- since we strip punctuation *before* tokenizing, the apostrophe is already gone by the time the tokenizer sees it. Order of preprocessing steps genuinely changes the result, not just a formality.

In [3]:
##stopwords
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english")) ##built in list of common words, they carry no signal for classification
def remove_stopwords(tokens):  ##takes a list of tokens (words) and returns essentially a filtered version of it
    return [word for word in tokens if word not in stop_words]

tokens = normalize_and_tokenize(sample)
print("Before stopword removal:", tokens)
print("After stopword removal", remove_stopwords(tokens))

Before stopword removal: ['chinas', 'xi', 'to', 'bring', 'large', 'ceo', 'delegation', 'on', 'us', 'visit', 'sources', 'say']
After stopword removal ['chinas', 'xi', 'bring', 'large', 'ceo', 'delegation', 'us', 'visit', 'sources', 'say']


Output cleaned up the common "buzzwords" into a cleaner set of words from the headline. One thing worth noting: in this headline "US" refers to United States, but since everything is lowercased it becomes "us" -- lowercasing collapsed a real, meaningful distinction. Also notice "us" survived stopword removal entirely -- NLTK's stopword list doesn't actually include "us" specifically (it has "we"/"our"/"ours", but not "us"), a real gap in the default list.

In [4]:
##Stemming vs lemmatization
from nltk.stem import PorterStemmer, WordNetLemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ["running", "studies", "cars", "delegation", "better"] ##sample
for word in test_words:
    print(f"{word}: stem={stemmer.stem(word)}, lemma={lemmatizer.lemmatize(word)}, lemma_verb={lemmatizer.lemmatize(word, pos='v')}")

running: stem=run, lemma=running, lemma_verb=run
studies: stem=studi, lemma=study, lemma_verb=study
cars: stem=car, lemma=car, lemma_verb=cars
delegation: stem=deleg, lemma=delegation, lemma_verb=delegation
better: stem=better, lemma=better, lemma_verb=better


Stemming is crude and rule-based -- it chops suffixes with fixed rules, no awareness of whether the result is a real word ("studies" -> "studi", "delegation" -> "deleg", neither a real English word). Lemmatization is smarter but needs to know the word's part of speech -- by default it assumes every word is a noun, which is why "running" stayed as "running" (as a noun, that already is its dictionary form) and only became "run" once explicitly told `pos="v"`. "better" didn't change either way since going to "good" requires knowing irregular comparative forms, beyond what either technique does.

In [5]:
##n-grams: n=1 is single words, n=2 is word pairs, etc -- same idea as week 2's bigram experiment, just written by hand this time
def get_ngrams(tokens, n):
    ##slides a window of n words across the token list, joining each window into one string ("large ceo" instead of two separate words)
    ##len(tokens) - n + 1 stops the window before it would run past the end of the list
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n +1)]

clean_tokens = remove_stopwords(normalize_and_tokenize(sample)) ##reusing both earlier functions on the same sample headline
print("Unigrams:", get_ngrams(clean_tokens, 1)) ##should look identical to clean_tokens, just single words
print("Bigrams:", get_ngrams(clean_tokens, 2)) ##adjacent word pairs -- expect one fewer entry than the unigram list

Unigrams: ['chinas', 'xi', 'bring', 'large', 'ceo', 'delegation', 'us', 'visit', 'sources', 'say']
Bigrams: ['chinas xi', 'xi bring', 'bring large', 'large ceo', 'ceo delegation', 'delegation us', 'us visit', 'visit sources', 'sources say']


Same underlying idea as Week 2 Day 5's bigram feature engineering experiment, just written out explicitly this time instead of letting `CountVectorizer(ngram_range=(1,2))` do it invisibly. We already know from that experiment that bigrams didn't help our tiny dataset -- most word pairs only show up once or twice, not enough repetition to learn from.

In [6]:
##bundles normalize + tokenize + stopword removal + lemmatize into one function, applied to every headline
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  ##default noun assumption, a known simplification
    return " ".join(tokens)  ##glues cleaned words back into one string, the format the vectorizer expects next

X_train_clean = X_train_text.apply(preprocess)  ##runs preprocess() across all 222 training headlines
X_test_clean = X_test_text.apply(preprocess)    ##same for all 56 test headlines

print("Original:", X_train_text.iloc[0])
print("Preprocessed:", X_train_clean.iloc[0])

Original: China's Xi to bring large CEO delegation on US visit, sources say
Preprocessed: china xi bring large ceo delegation u visit source say


Real example of the lemmatization limitation showing up: "us" became "u" -- the lemmatizer's default noun assumption treated it like a regular plural noun (the same way "sources" correctly became "source") and tried stripping the "s", producing a nonsensical singular "u". A concrete downside of the simplification, not just a hypothetical.

In [7]:
##the actual new material: TF-IDF instead of plain word counts, on the cleaned text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

tfidf = TfidfVectorizer()  ##downweights words common across many headlines, upweights rare/distinctive ones -- unlike CountVectorizer, which treats both the same
X_train_tfidf = tfidf.fit_transform(X_train_clean)  ##fit only on train, same rule we've followed since week 2
X_test_tfidf = tfidf.transform(X_test_clean)

print("TF-IDF matrix shape:", X_train_tfidf.shape)

##LogisticRegression on purpose -- same model as week 2 day 2's original CountVectorizer baseline (61%), for a direct, clean comparison
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train_labels)
predictions = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_labels, predictions))
print(classification_report(y_test_labels, predictions, zero_division=0))

TF-IDF matrix shape: (222, 1148)
Accuracy: 0.625
              precision    recall  f1-score   support

    Business       0.53      1.00      0.69        17
      Energy       0.00      0.00      0.00         3
      Health       0.00      0.00      0.00         3
     Markets       0.75      0.75      0.75        20
    Politics       0.00      0.00      0.00         2
  Technology       0.75      0.27      0.40        11

    accuracy                           0.62        56
   macro avg       0.34      0.34      0.31        56
weighted avg       0.58      0.62      0.56        56



TF-IDF has two parts multiplied together: TF (term frequency, basically what CountVectorizer already gives -- how often a word shows up in this one headline) and IDF (inverse document frequency, which downweights words common across *many* headlines and upweights rare/distinctive ones). CountVectorizer treats a super common word and a rare distinctive word the same; TF-IDF is supposed to fix that.

62.5% accuracy, a bit above Week 2 Day 2's original 61% baseline -- though that's not a clean comparison, since this run uses the expanded 278-row dataset, not the original 111-row one. Energy, Health, and Politics all landed at 0.00 precision/recall/f1 again -- still too few examples (3, 3, 2 in this split) for the model to learn anything reliable about them, same story as always.

In [8]:
##controlled comparison: same cleaned text, same split, same model -- only the vectorizer changes (plain counts instead of TF-IDF)
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer()
X_train_counts = count_vec.fit_transform(X_train_clean)
X_test_counts = count_vec.transform(X_test_clean)

count_model = LogisticRegression(max_iter=1000)
count_model.fit(X_train_counts, y_train_labels)
count_predictions = count_model.predict(X_test_counts)

print("\nCountVectorizer accuracy (same data, same model):", accuracy_score(y_test_labels, count_predictions))


CountVectorizer accuracy (same data, same model): 0.7142857142857143


## Takeaway

Genuinely surprising result: **CountVectorizer beat TF-IDF** on the exact same cleaned data, same split, same model -- 71.4% vs. 62.5%. The opposite of what's commonly assumed about TF-IDF always being the better choice.

A plausible explanation: TF-IDF's downweighting of "common" words is based on document frequency across only 222 training headlines -- a small enough corpus that this frequency estimate is pretty noisy. It's also possible some of the words TF-IDF downweighted for being common (like "stock") are actually strong, reliable category signals *within this specific dataset*, even though they show up often. TF-IDF's normalization can also affect short texts like headlines more than longer documents, since there's less redundancy for the weighting to meaningfully act on.

This connects to the same theme that's shown up all project long -- a theoretically more sophisticated technique doesn't automatically win, especially on a small dataset. XGBoost lost to plain Gradient Boosting (Week 2 Day 6), `RandomizedSearchCV` picked a worse model than `GridSearchCV` despite a similar CV score (Week 2 Day 5), and now TF-IDF loses to plain word counts. Worth checking the assumption with real held-out data every time, not just trusting which technique sounds more advanced.